# H0 · Reference Ship Database: First Numbers from Similar Ships
**Ship and Ocean Systems Design 2026 · optional, not graded · about 1 hour**

**Objectives**
1. Use a regression formula, y = a·xᵇ, in Python.
2. Repeat Exercise 1.4 in code: the reference tanker, a deadweight carrier.
3. Size a container ship from its TEU: the volume route of your project.
4. Fit your own regression to a list of ships: the start of D2.

Sources: Lecture 1.4; Papanikolaou (2026) Appendix A, Table 2.15.

**How to run:** in Google Colab, first *File › Save a copy in Drive* (so your work is kept), then run the cells from the top with Shift+Enter. Change only the lines marked `# INPUT`.

## 1 · A regression in Python
Most Appendix A fits have the form y = a·xᵇ. One small function does them all.

In [ ]:
def fit(a, b, x):
    """Regression y = a * x**b, as printed on the Appendix A figures."""
    return a * x**b

print(fit(2.0, 0.5, 100))   # test: 2 * 100**0.5 = 20

## 2 · Deadweight carrier: the reference ship (Exercise 1.4)
Δ from the deadweight, then L, B, T, D from the tanker fits, and C_B from Horn's formula.

In [ ]:
import math

DWT = 37600      # INPUT: deadweight [t]
V = 14.5         # INPUT: service speed [kn]
rho = 1.025      # sea water [t/m3]

delta = DWT / fit(0.541223, 0.0390339, DWT)   # Fig. A.51: DWT/Delta
L = fit(6.86408, 0.307233, DWT)               # Fig. A.53
B = fit(0.962394, 0.324898, DWT)              # Fig. A.54
T = fit(0.45011, 0.303134, DWT)               # Fig. A.56
D = fit(0.45914, 0.330623, DWT)               # Fig. A.55

Fn = V * 0.5144 / math.sqrt(9.81 * L)         # Froude number
CB = 1.06 - 1.68 * Fn                         # Horn, Table 2.15

print(f"Delta = {delta:,.0f} t")
print(f"L = {L:.1f} m, B = {B:.2f} m, T = {T:.2f} m, D = {D:.2f} m")
print(f"Fn = {Fn:.3f}, CB = {CB:.3f}")

**Check:** does the hull float at that Δ? Buoyancy is ρ·C_B·L·B·T.

In [ ]:
buoyancy = rho * CB * L * B * T
gap = 100 * (buoyancy / delta - 1)
print(f"Buoyancy = {buoyancy:,.0f} t, gap = {gap:+.1f} %")

## 3 · Volume carrier: a container ship from its TEU
The size comes from the boxes. Steps: TEU → cargo (10 t per TEU, as in the project brief) → DWT (cargo = 0.75·DWT, Fig. A.49) → the container fits of Appendix A.3.
Example: the 2,000 TEU ship of Lecture 1.4, slide 9.

In [ ]:
TEU = 2000        # INPUT: number of containers
V = 20.3          # INPUT: service speed [kn] (Fig. A.50 for this length)

cargo = 10 * TEU                 # t
DWT = cargo / 0.75               # t
print(f"Cargo = {cargo:,.0f} t, DWT = {DWT:,.0f} t")
if DWT < 4000 or DWT > 105000:
    print("Warning: outside the data of the container fits (4,000-105,000 t)")

delta = fit(1.77955, 0.975578, DWT)          # Fig. A.30
L = fit(3.54132, 0.388442, DWT)              # Fig. A.31
B = fit(1.55219, 0.284381, DWT)              # Fig. A.32
D = fit(0.299394, 0.38902, DWT)              # Fig. A.33
T = -17.1581 + 2.72338 * math.log(DWT)       # Fig. A.34 (a log fit)
CB = fit(0.908683, -0.0320423, DWT)          # Fig. A.38

print(f"Delta = {delta:,.0f} t")
print(f"L = {L:.1f} m, B = {B:.2f} m, T = {T:.2f} m, D = {D:.2f} m, CB = {CB:.3f}")

In [ ]:
buoyancy = rho * CB * L * B * T
Fn = V * 0.5144 / math.sqrt(9.81 * L)
print(f"Buoyancy gap = {100 * (buoyancy / delta - 1):+.1f} %")
print(f"CB from the fit = {CB:.3f}, CB from Horn = {1.06 - 1.68 * Fn:.3f}")

## 4 · Your own regression (start of D2)
Put your similar ships in the two lists, then fit L = a·TEUᵇ. The lists below hold points read from Kristensen's Fig. A.47 (Lecture 1.4), only so the cell runs: replace them with your ships.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# INPUT: one value per ship, same order in both lists
teu_list = [210, 420, 662, 1072, 1471, 1976, 2459, 2974, 3600]
L_list   = [80, 100, 120, 140, 160, 180, 200, 220, 240]

# A straight line through the logarithms gives b and log(a)
b, log_a = np.polyfit(np.log(teu_list), np.log(L_list), 1)
a = math.exp(log_a)
print(f"L = {a:.3f} * TEU^{b:.3f}")
print(f"For {TEU} TEU: L = {fit(a, b, TEU):.1f} m")

x = np.linspace(min(teu_list), max(teu_list), 100)
plt.plot(teu_list, L_list, "o", label="ships")
plt.plot(x, fit(a, b, x), "-", label="fit")
plt.xlabel("TEU"); plt.ylabel("L [m]"); plt.legend(); plt.grid(True)
plt.show()

## Questions · answer in one or two lines each
**Q1** Section 2: how large is the buoyancy gap for the tanker? Which value would you change first? *(Lecture 1.5, slides 3 to 6, closes it.)*

**Q2** Section 3: compare L from the fit with the 182 m read from Kristensen's curve in Lecture 1.4, slide 9. Is the difference a problem?

**Q3** Put your project ship's TEU in section 3. Is it inside the data range? If not, what can you use instead?

*Your answers:*
